<a href="https://colab.research.google.com/github/Junhojuno/pytorch-tutorial/blob/master/Make_VGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### VGG code를 이해해보자

In [1]:
import torch.nn as nn
import torch.utils.model_zoo as model_zoo

In [2]:
__all__ = [
    'VGG', 'vgg11', 'vgg11_bn', 'vgg13', 'vgg13_bn', 'vgg16', 'vgg16_bn',
    'vgg19_bn', 'vgg19',
]


# imagenet 데이터를 pre-training시켜놓은 모델
model_urls = {
    'vgg11': 'https://download.pytorch.org/models/vgg11-bbd30ac9.pth',
    'vgg13': 'https://download.pytorch.org/models/vgg13-c768596a.pth',
    'vgg16': 'https://download.pytorch.org/models/vgg16-397923af.pth',
    'vgg19': 'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth',
    'vgg11_bn': 'https://download.pytorch.org/models/vgg11_bn-6002323d.pth',
    'vgg13_bn': 'https://download.pytorch.org/models/vgg13_bn-abd245e5.pth',
    'vgg16_bn': 'https://download.pytorch.org/models/vgg16_bn-6c64b313.pth',
    'vgg19_bn': 'https://download.pytorch.org/models/vgg19_bn-c79401a0.pth',
}

In [3]:
class VGG(nn.Module):

    def __init__(self, features, num_classes=1000, init_weights=True):
        super(VGG, self).__init__()
        self.features = features # couvolution layer를 의미한다.
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
       
        # self.classifier가 fully connected layer를 의미
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        x = self.features(x) # convolution layer
        x = self.avgpool(x) # average pooling
        x = x.view(x.size(0), -1)
        x = self.classifier(x) # FC layer
        return x

    def _initialize_weights(self):
        for m in self.modules(): # self.modules는 뭐지? nn.Module에 있는건가?
            if isinstance(m, nn.Conv2d): # 만약 m이 nn.Conv2d일때를 의미
                # kaiming normalization은 activation function에 따라 초기화를 잘해줄수 있다?
                # activation function에 따라 최적의 초기화방식이 다른가 보다?!
                # 여기선 relu
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0) # bias는 0으로 세팅
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1) # batch norm의 weight는 전부 1
                nn.init.constant_(m.bias, 0) # batch norm의 bias는 전부 0
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [5]:
# feature를 어떻게 만들어서 넣어줄 것이냐
# 예를 들어 cfg['A']로 진행 
#  'A': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

def make_layers(cfg, batch_norm=False):
    layers = []
    in_channels = 3 
    for v in cfg:
        if v == 'M': # v는 처음이 64, 맨 마지막이 'M' 이겠지?! yes!
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)

##### make_layers를 이해해보자
- model = make_layers(cfg['A'], batch_norm=False)로 했을때 
- 어떤 모델이 생성될지를 확인해보자
- 아래가 순서대로 nn.Sequential에 들어간다.

```python

<첫번째. 64가 들어갔을때>
conv2d = nn.Conv2d(3, 64, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<두번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<세번째. 128>
conv2d = nn.Conv2d(64, 128, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<네번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<다섯번째. 256>
conv2d = nn.Conv2d(128, 256, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<여섯번째. 256>
conv2d = nn.Conv2d(256, 256, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<일곱번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<여덟번째. 512>
conv2d = nn.Conv2d(256, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<아홉번째. 512>
conv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<열번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<열한번째. 512>
conv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<열두번째. 512>
conv2d = nn.Conv2d(512, 512, kernel_size=3, padding=1)
nn.ReLU(inplace=True)

<열세번째. 'M'>
nn.MaxPool2d(kernel_size=2, stride=2)

<return은...>
# convolution layer 8개
nn.Sequential(
              nn.Conv2d(3, 64, kernel_size=3, padding=1) # 1
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(64, 128, kernel_size=3, padding=1) # 2
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(128, 256, kernel_size=3, padding=1) # 3
             ,nn.ReLU(inplace=True)
             ,nn.Conv2d(256, 256, kernel_size=3, padding=1) # 4
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(256, 512, kernel_size=3, padding=1) # 5
             ,nn.ReLU(inplace=True)
             ,nn.Conv2d(512, 512, kernel_size=3, padding=1) # 6
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
             ,nn.Conv2d(512, 512, kernel_size=3, padding=1) # 7
             ,nn.ReLU(inplace=True)
             ,nn.Conv2d(512, 512, kernel_size=3, padding=1) # 8
             ,nn.ReLU(inplace=True)
             ,nn.MaxPool2d(kernel_size=2, stride=2)
)
```

In [6]:
# 'custom'추가도 가능하다.
# 각 숫자의 의미는 channerl입니다. 헷갈리지 맙시다.
cfg = {
    'A': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'], # convolution 8 + fc 3 = vgg11
    'B': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'], # 10 + 3 = vgg13
    'D': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'], # 13 + 3 = vgg16
    'E': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'], # 16 + 3 = vgg19
    'custom' : [64,64,64,'M',128,128,128,'M',256,256,256,'M']
}

In [7]:
# 비교해보자
# 직접 계산한거랑 맞는지
conv = make_layers(cfg=cfg['A'])
conv

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU(inplace)
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU(inplace)
  (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU(inplace)
  (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): ReLU(inplace)
  (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (14): ReLU(inplace)
  (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (16): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17)

In [8]:
# batch_norm=True로 놓으면
# 각 convolution아래 batch norm이 추가된 것을 볼 수 있다.
conv2 = make_layers(cfg['A'], batch_norm=True)
conv2

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace)
  (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU(inplace)
  (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU(inplace)
  (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (13): ReLU(inplace)
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (15)

In [9]:
# 실제 VGG를 만들어보자
# class 갯수는 cifar10인 경우 10개
# fc layer에서 받는 in_feature의 수가 고정되어있기 때문에
# custom VGG사용하려면 이부분을 적절하게 수정해줘야한다.
model = VGG(features=make_layers(cfg['A']), num_classes=10, init_weights=True)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (16): Conv2d(512, 512, kern

### VGG를 CIFAR10에 적용해보자!!!

In [10]:
import torch
import torch.nn as nn

import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [12]:
import visdom
# import 전에 사전에 cmd창에 "python -m visdom.server"를 입력하여 server를 연결해두자 (선 연결 필수)
vis = visdom.Visdom()
vis.close(env='main')

''

In [13]:
# define loss tracker
# 10.3 visdom 강의 참고
def loss_tracker(loss_plot, loss_value, num):
    vis.line(X=num, Y=loss_value, win=loss_plot, update='append')

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device : ",device)
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed(777)

device :  cpu


In [16]:
# transforms.Compose() --> 공통으로 적용할 transform을 묶는다.
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))]) # RGB라서 3개인듯

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset=trainset, batch_size=512, shuffle=True, num_workers=4)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(dataset=testset, batch_size=4, shuffle=False, num_workers=4)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse','ship', 'truck')

100%|█████████▉| 170319872/170498071 [01:23<00:00, 1572875.02it/s]

Files already downloaded and verified


170500096it [01:40, 1572875.02it/s]                               

In [28]:
# (image갯수, height, width, channel) --> 왜 tensorflow순서로 나오지?? transform이 안됐네
# train_loader.dataset.data 자체는 Dataloader에 들어온 data의 shape을 의미하는가보다..(transforms되기 전의 데이터 shape)
train_loader.dataset.data.shape 

(50000, 32, 32, 3)

In [30]:
# (batch size, channels, heights, width)
# input image의 shape : 32 x 32
for x,y in train_loader:
    print(x.shape)
    break

torch.Size([512, 3, 32, 32])


In [24]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

def imshow(img):
    img = img / 2 + 0.5 # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1,2,8)))
    plt.show()
    
# get some random training images
# visdom에 batch size만큼의 이미지 띄우기
dataiter = iter(train_loader)
images, labels = dataiter.next()
vis.images(images/2 + 0.5)

'window_373cbb757d1f7c'

##### Let's go VGG

In [25]:
import torchvision.models.vgg as vgg

In [26]:
# custom convolution network
cfg = [32,32,'M', 64,64,128,128,128,'M', 256,256,256,512,512,512,'M'] # 13 convolution + 3 fc = vgg16

In [32]:
# 이걸 통과하면 output의 shape : [1, 512, 4, 4]
cnn = vgg.make_layers(cfg=cfg)
sample = torch.Tensor(1,3,32,32)
out = cnn(sample)
out.shape

torch.Size([1, 512, 4, 4])

In [33]:
class VGG(nn.Module):

    def __init__(self, features, num_classes=1000, init_weights=True):
        super(VGG, self).__init__()
        self.features = features # couvolution layer를 의미한다.
#         self.avgpool = nn.AdaptiveAvgPool2d((7, 7)) --> convolution 통과하면 image가 4 x 4로 작아지기때문에 kernel size 7의 pooling이 안됨.
       
        # self.classifier가 fully connected layer를 의미
        self.classifier = nn.Sequential(
            nn.Linear(512 * 4 * 4, 4096), # --> 원래 nn.Linear(512 * 7 * 7, 4096)인데 custom convolution output에 맞게 수정
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        x = self.features(x) # convolution layer
#         x = self.avgpool(x) # average pooling
        x = x.view(x.size(0), -1)
        x = self.classifier(x) # FC layer
        return x

    def _initialize_weights(self):
        for m in self.modules(): # self.modules는 뭐지? nn.Module에 있는건가?
            if isinstance(m, nn.Conv2d): # 만약 m이 nn.Conv2d일때를 의미
                # kaiming normalization은 activation function에 따라 초기화를 잘해줄수 있다?
                # activation function에 따라 최적의 초기화방식이 다른가 보다?!
                # 여기선 relu
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0) # bias는 0으로 세팅
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1) # batch norm의 weight는 전부 1
                nn.init.constant_(m.bias, 0) # batch norm의 bias는 전부 0
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [34]:
vgg16 = VGG(vgg.make_layers(cfg), num_classes=10, init_weights=True).to(device)

In [37]:
# 최종적으로 output이 잘나오는지 테스트해보자
a = torch.Tensor(1,3,32,32).to(device)
out = vgg16(a)
out.shape # 잘 나옴.

torch.Size([1, 10])

In [38]:
# 모델 다 만들었으니
# loss, optimizer 만들자
cost_func = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(vgg16.parameters(), lr=0.005, momentum=0.9)

# epoch 50번을 돌리는 동안
# 5번에 한번씩 learning rate에 0.9를 곱해주세요! 의미
lr_schedule = optim.lr_scheduler.StepLR(optimizer=optimizer, gamma=0.9, step_size=5)

In [39]:
# make plot
loss_plt = vis.line(Y=torch.Tensor(1).zero_(), opts=dict(title='loss_tracker', legend=['loss'], showlegend=True))

In [ ]:
# training
print(len(train_loader))
epochs = 50

for epoch in range(epochs):
    running_loss = 0.0
    lr_schedule.step() # 이게 5회실행되면 learning rate scheduling : 0.005 * 0.9 * 0.9 * .....
    for i, data in enumerate(train_loader, 0): # 여기서 0은 index를 0부터 시작하세요를 의미하는 듯?
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        hypothesis = vgg16(inputs)
        cost = cost_func(hypothesis, labels)
        cost.backward()
        optimizer.step()
        
        # print statistics
        running_loss += cost.item()
        if i % 30 == 29:
            # len(train_loader) == total batch == CIFAR10 image갯수 / batch size
            loss_tracker(loss_plot=loss_plt, loss_value=torch.Tensor([running_loss/30]), num=torch.Tensor([i+epoch*len(train_loader)]))
            print('[%d, %5d] loss : %.3f' % (epoch+1, i+1, running_loss / 30))
            running_loss = 0.0
print('Training Finished..!')